In [17]:
import nb_setup
import os
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import get_linear_schedule_with_warmup
from collections import defaultdict
from sklearn.metrics import f1_score

from src.data import load_tokenised_data
from src.answerability.evaluation import evaluate_answerability_predictions
from src.integrated_qa import (
    build_qa_features,
    QATrainDataset,
    train_collate_fn,
    val_collate_fn,
    f1_score_squad,
    exact_match_score
)
from evaluate import load as load_metric

In [2]:
def enforce_reproducibility(seed=42):
    # Sets seed manually for both CPU and CUDA
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For atomic operations there is currently
    # no simple way to enforce determinism, as
    # the order of parallel operations is not known.
    # CUDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # System based
    random.seed(seed)
    np.random.seed(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

enforce_reproducibility()

df_train_tokenised, df_val_tokenised = load_tokenised_data()
df_train_subset = df_train_tokenised.sample(n=400, random_state=42).reset_index(drop=True)

In [3]:
squad_metric = load_metric("squad_v2")

def compute_squad(*, references, predictions):
    return squad_metric.compute(references=references, predictions=predictions)

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 384
DOC_STRIDE = 128

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
)

# Freeze all XLM-R encoder parameters
for param in model.roberta.parameters():
    param.requires_grad = False

# Unfreeze last 4 Transformer layers
for layer in model.roberta.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Keep QA head trainable
for param in model.qa_outputs.parameters():
    param.requires_grad = True

model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_features = build_qa_features(
    df_train_tokenised,
    # df_train_subset,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

validation_features = build_qa_features(
    df_val_tokenised,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

train_dataset = QATrainDataset(train_features)

train_dl = DataLoader(
    train_dataset,
    collate_fn=train_collate_fn,
    shuffle=True,
    batch_size=4,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
qa_outputs.bias             | MISSING    | 
qa_outputs.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
lr = 2e-5
n_epochs = 10
weight_decay = 0.01
warmup_steps = 200

no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

optimizer_grouped_parameters = [
    {
        "params": [
            p
            for n, p in model.named_parameters()
            if p.requires_grad
            and not any(nd in n for nd in no_decay)
        ],
        "weight_decay": weight_decay,
    },
    {
        "params": [
            p
            for n, p in model.named_parameters()
            if p.requires_grad
            and any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=n_epochs * len(train_dl),
)

total_steps = n_epochs * len(train_dl)

print("Steps per epoch:", len(train_dl))
print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)

Steps per epoch: 1629
Total steps: 16290
Warmup steps: 200


In [6]:
def train(model, train_dl, optimizer, scheduler, n_epochs, device):
    losses = []
    for epoch in range(n_epochs):
        loss_epoch = []
        model.train()
        for batch in tqdm(train_dl):
            optimizer.zero_grad()

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            loss = outputs.loss

            losses.append(loss.item())
            loss_epoch.append(loss.item())

            loss.backward()

            optimizer.step()
            scheduler.step()

        print(
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"Loss: {np.mean(loss_epoch):.4f}"
        )

    return losses

losses = train(model, train_dl, optimizer, scheduler, n_epochs, device)

100%|██████████| 1629/1629 [07:22<00:00,  3.68it/s]


Epoch 1/10 | Loss: 2.9157


100%|██████████| 1629/1629 [07:13<00:00,  3.76it/s]


Epoch 2/10 | Loss: 2.0726


100%|██████████| 1629/1629 [08:16<00:00,  3.28it/s]


...

Epoch 8/10 | Loss: 0.7871


100%|██████████| 1629/1629 [07:03<00:00,  3.84it/s]


Epoch 9/10 | Loss: 0.7200


100%|██████████| 1629/1629 [06:45<00:00,  4.01it/s]

Epoch 10/10 | Loss: 0.6523


In [21]:
load_model = False
os.makedirs("../checkpoints", exist_ok=True)
SAVE_PATH = "../checkpoints/integrated_qa"

if load_model:
    tokenizer = AutoTokenizer.from_pretrained(
        SAVE_PATH,
        use_fast=True
    )

    model = AutoModelForQuestionAnswering.from_pretrained(
        SAVE_PATH,
        attn_implementation="eager",
    ).to(device)

else:
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
def get_validation_features(df, tokenizer, max_length=384, doc_stride=128):
    samples = df.reset_index(drop=True)

    batch = tokenizer(
        samples["question"].tolist(),
        samples["context"].tolist(),
        padding="max_length",
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    sample_mapping = batch.pop("overflow_to_sample_mapping")

    batch["example_index"] = []

    for i in range(len(batch["input_ids"])):

        sample_idx = sample_mapping[i]
        sequence_ids = batch.sequence_ids(i)

        batch["example_index"].append(sample_idx)

        # Keep offsets only for context tokens
        batch["offset_mapping"][i] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(
                batch["offset_mapping"][i]
            )
        ]

    features = []

    for i in range(len(batch["input_ids"])):

        features.append(
            {
                "input_ids": batch["input_ids"][i],
                "attention_mask": batch["attention_mask"][i],
                "offset_mapping": batch["offset_mapping"][i],
                "example_index": batch["example_index"][i],
            }
        )

    return features


validation_features = get_validation_features(df_val_tokenised, tokenizer, max_length=MAX_LENGTH, doc_stride=DOC_STRIDE)

val_dl = DataLoader(validation_features, collate_fn=val_collate_fn, batch_size=32)

In [8]:
def predict(
    model,
    valid_dl,
    device,
):
    model.eval()

    start_logits_all = []
    end_logits_all = []

    with torch.no_grad():

        for batch in tqdm(valid_dl, desc="Evaluation"):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])

            start_logits_all.extend(
                outputs.start_logits
                .detach()
                .cpu()
                .numpy()
            )

            end_logits_all.extend(
                outputs.end_logits
                .detach()
                .cpu()
                .numpy()
            )

    return (start_logits_all, end_logits_all)


def post_process_predictions(
    examples,
    dataset,
    logits,
    threshold=0.0,
    num_possible_answers=20,
    max_answer_length=30,
):

    all_start_logits, all_end_logits = logits

    features_per_example = defaultdict(list)

    for i, feature in enumerate(dataset):
        features_per_example[feature["example_index"]].append(i)

    predictions = []

    for example_idx, sample in examples.reset_index(drop=True).iterrows():

        feature_indices = features_per_example[example_idx]

        context = sample["context"]

        best_span_score = -np.inf
        best_span_text = ""

        best_null_score = np.inf

        for feature_idx in feature_indices:

            start_logits = all_start_logits[feature_idx]

            end_logits = all_end_logits[feature_idx]

            offset_mapping = dataset[feature_idx]["offset_mapping"]

            # CLS = no answer
            null_score = start_logits[0] + end_logits[0]

            best_null_score = min(best_null_score, null_score)

            start_indices = np.argsort(start_logits)[::-1][:num_possible_answers]

            end_indices = np.argsort(end_logits)[::-1][:num_possible_answers]

            for start_index in start_indices:
                for end_index in end_indices:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue

                    if (
                        start_index > end_index
                        or end_index - start_index + 1
                        > max_answer_length
                    ):
                        continue

                    score = start_logits[start_index] + end_logits[end_index]

                    if score > best_span_score:
                        start_char = offset_mapping[start_index][0]

                        end_char = offset_mapping[end_index][1]

                        best_span_score = score

                        best_span_text = context[start_char:end_char]

        no_answer_score = best_null_score - best_span_score

        prediction = (
            ""
            if no_answer_score > threshold
            else best_span_text
        )

        predictions.append(
            {
                "example_index": example_idx,
                "prediction": prediction,
                "no_answer_score": no_answer_score,
            }
        )

    return pd.DataFrame(predictions)

logits = predict(model, val_dl, device)

Evaluation: 100%|██████████| 38/38 [00:55<00:00,  1.47s/it]


In [9]:
predictions = post_process_predictions(
    df_val_tokenised,
    validation_features,
    logits,
    threshold=np.inf,
)
predictions.head()

,example_index,prediction,no_answer_score
0,0,University of Helsinki,-18.769796
1,1,"Finland, Sweden, Denmark, Estonia, Latvia, Lit...",-27.038319
2,2,The King's School,-10.781025
3,3,doubtful,5.202164
4,4,Abdrabbuh Mansur Hadi,-23.606853


In [10]:
val_examples = df_val_tokenised.reset_index(drop=True)

predictions["gold_answer"] = predictions["example_index"].map(
    lambda i: (
        str(val_examples.loc[i, "answer"])
        if val_examples.loc[i, "answerable"]
        else ""
    )
)

predictions["gold_answerable"] = predictions["example_index"].map(
    lambda i: int(val_examples.loc[i, "answerable"])
)

predictions["lang"] = predictions["example_index"].map(
    lambda i: val_examples.loc[i, "lang"]
)

In [11]:
threshold_results = []

thresholds = np.linspace(
    predictions["no_answer_score"].min(),
    predictions["no_answer_score"].max(),
    100,
)

for threshold in thresholds:

    tuned_predictions = np.where(
        predictions["no_answer_score"] > threshold,
        "",
        predictions["prediction"],
    )

    em = np.mean([
        exact_match_score(pred, gold)
        for pred, gold in zip(tuned_predictions, predictions["gold_answer"])
    ])

    token_f1 = np.mean([
        f1_score_squad(pred, gold)
        for pred, gold in zip(tuned_predictions, predictions["gold_answer"])
    ])

    threshold_results.append(
        {
            "threshold": threshold,
            "EM": em,
            "Token F1": token_f1,
        }
    )

threshold_results = pd.DataFrame(threshold_results)

In [12]:
best_row = threshold_results.loc[threshold_results["Token F1"].idxmax()]

best_threshold = best_row["threshold"]

print(best_row)

threshold   -5.895390
EM           0.490043
Token F1     0.583578
Name: 64, dtype: float64


In [13]:
predictions["prediction_tuned"] = np.where(
    predictions["no_answer_score"] > best_threshold,
    "",
    predictions["prediction"],
)

In [14]:
predictions["EM"] = [
    exact_match_score(pred, gold)
    for pred, gold in zip(predictions["prediction_tuned"], predictions["gold_answer"])
]

predictions["Token F1"] = [
    f1_score_squad(pred, gold)
    for pred, gold in zip(predictions["prediction_tuned"], predictions["gold_answer"])
]

predictions["pred_answerable"] = (predictions["prediction_tuned"].str.len() > 0).astype(int)

xlmr_answerability_results = evaluate_answerability_predictions(
    y_true=predictions["gold_answerable"].astype(int),
    y_pred=predictions["pred_answerable"],
    languages=predictions["lang"],
    model_name="Integrated XLM-R QA",
)

In [15]:
print("SQuAD-v2 Exact Match:", predictions["EM"].mean())
print("SQuAD-v2 Token F1:", predictions["Token F1"].mean())

display(xlmr_answerability_results[["Language", "N", "Macro F1"]])

SQuAD-v2 Exact Match: 0.49004329004329006
SQuAD-v2 Token F1: 0.5835778036994563


,Language,N,Macro F1
0,Overall,1155,0.831155
1,Arabic,415,0.905479
2,Korean,356,0.821285
3,Telugu,384,0.766617
